# Laboratorio 7 – Regresión Logística

## 0. Configuración — Pipeline de Labs 4–6

In [24]:
import pyreadr, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import time, warnings, tracemalloc, cProfile, pstats, io
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
    StratifiedKFold, learning_curve, GridSearchCV)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, log_loss)
from sklearn.pipeline import Pipeline
from scipy import stats
import statsmodels.api as sm

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

In [25]:
# Pipeline idéntico a Labs 4–6
result = pyreadr.read_r('listings.Rdata')
df_raw = result[list(result.keys())[0]].copy()

df = df_raw.copy()
if df['price'].dtype == object:
    df['price'] = (df['price'].str.replace(r'[\$,]','',regex=True)
                   .str.strip().replace('',np.nan).astype(float))
q_high = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= q_high)].copy()

cols_drop = ['id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_thumbnail_url','host_picture_url','host_about','host_verifications',
    'amenities','calendar_updated','calendar_last_scraped','license',
    'bathrooms_text','minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights',
    'minimum_nights_avg_ntm','maximum_nights_avg_ntm']
df = df.drop(columns=[c for c in cols_drop if c in df.columns])
null_pct = df.isnull().mean()
df = df.drop(columns=null_pct[null_pct > 0.60].index.tolist())

if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_years'] = ((pd.Timestamp('2024-01-01') - df['host_since']).dt.days/365).round(1)
    df = df.drop(columns=['host_since'])
df = df.drop(columns=[c for c in ['first_review','last_review'] if c in df.columns], errors='ignore')

for col in ['host_is_superhost','host_has_profile_pic','host_identity_verified',
            'has_availability','instant_bookable']:
    if col in df.columns:
        df[col] = df[col].map({'t':1,'f':0,True:1,False:0})
for col in ['host_response_rate','host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].str.replace('%','',regex=False).str.strip().astype(float, errors='ignore')

TARGET = 'price'
num_features = [c for c in df.select_dtypes(include='number').columns if c != TARGET]
cat_features = [c for c in ['room_type','property_type','neighbourhood_cleansed',
                              'host_response_time'] if c in df.columns]
for col in cat_features:
    freq = df[col].value_counts(normalize=True)
    df[col] = df[col].replace(freq[freq < 0.01].index, 'Otro')
df_encoded = pd.get_dummies(df[num_features+cat_features+[TARGET]],
                             columns=cat_features, drop_first=True, dtype=int)
for col in df_encoded.columns:
    if df_encoded[col].isnull().any():
        df_encoded[col].fillna(df_encoded[col].median(), inplace=True)

# Variables categóricas de precio (igual que Labs anteriores)
p33 = df_encoded[TARGET].quantile(0.33)  # $140
p67 = df_encoded[TARGET].quantile(0.67)  # $267
df_encoded['price_category'] = df_encoded[TARGET].apply(
    lambda p: 'Económico' if p<=p33 else ('Intermedio' if p<=p67 else 'Caro'))

feature_cols = [c for c in df_encoded.columns if c not in [TARGET,'price_category']]
X = df_encoded[feature_cols]
y_price = df_encoded[TARGET]

# Split IDÉNTICO (random_state=42, test_size=0.20) — mismo de Labs 4-6
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.20, random_state=42)

print(f"Dataset: {df_encoded.shape[0]:,} filas × {len(feature_cols)} features")
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"P33=${p33:.0f}  P67=${p67:.0f}")

Dataset: 75,531 filas × 73 features
Train: 60,424  |  Test: 15,107
P33=$140  P67=$267


## Actividad 1 – Variables Dicotómicas

Se crean tres variables binarias (0/1) a partir de `price_category`, una por cada categoría.
Estas variables permiten entrenar un clasificador binario independiente para cada categoría.
Se enfoca en es_cara

- es_cara = 1 si precio > $267/noche (percentil 67), 0 en caso contrario
- es_intermedia = 1 si $140 < precio ≤ $267 (entre P33 y P67), 0 en otro caso
- es_economica = 1 si precio ≤ $140/noche (percentil 33), 0 en caso contrario


In [26]:
# La variable price_category ya fue creada en Labs anteriores.
# Aquí convertimos cada categoría en una variable binaria independiente.

df_encoded['es_cara']       = (df_encoded['price_category'] == 'Caro').astype(int)
df_encoded['es_intermedia'] = (df_encoded['price_category'] == 'Intermedio').astype(int)
df_encoded['es_economica']  = (df_encoded['price_category'] == 'Económico').astype(int)

# Verificar distribución
for var in ['es_cara','es_intermedia','es_economica']:
    n1 = df_encoded[var].sum()
    n0 = len(df_encoded) - n1
    print(f"{var:<15}: {n1:>6} positivos ({n1/len(df_encoded)*100:.1f}%)  |  {n0:>6} negativos ({n0/len(df_encoded)*100:.1f}%)")

print()
# Verificar que son mutuamente excluyentes y exhaustivas
assert (df_encoded['es_cara']+df_encoded['es_intermedia']+df_encoded['es_economica']).eq(1).all()
print("Las tres variables son mutuamente excluyentes y exhaustivas (suma siempre = 1).")
print()
print("Clase de interés: es_cara, detectar propiedades con precio > $267/noche")


es_cara        :  24809 positivos (32.8%)  |   50722 negativos (67.2%)
es_intermedia  :  25787 positivos (34.1%)  |   49744 negativos (65.9%)
es_economica   :  24935 positivos (33.0%)  |   50596 negativos (67.0%)

Las tres variables son mutuamente excluyentes y exhaustivas (suma siempre = 1).

Clase de interés: es_cara, detectar propiedades con precio > $267/noche
